# Optimus 5-Prime for 5′ UTR design

[Notebook](single-task.ipynb) · [Installation](../docs/installation.md)

Maximize the predicted mean ribosome load of a 50-nt 5′ UTR. Reuse the 100 starting sequences and exclude runs of six identical bases, as in the Extended Data analysis. Install MEME Suite with the notebook dependencies. The first model load downloads the pretrained checkpoint.

## 1. Load the model



In [1]:
import re
from pathlib import Path
from tempfile import mkdtemp

import pandas as pd
import torch
from bioseq_models import load_model
from MotifSift import call_motifs, generate_designs, select_candidates

device = "cuda" if torch.cuda.is_available() else "cpu"
model = load_model("optimus5prime", device=device)
seeds = pd.read_csv("data/optimus_initial_sequences.csv")["sequence"].tolist()

run = Path(mkdtemp(prefix="motifsift-example-"))




## 2. Generate candidates

Run the eleven optimizers and keep 100 candidates per optimizer. The saved tables show the previous full run. New runs use the current MotifSift defaults.



In [2]:
def no_homopolymer6(sequences):
    return [re.search(r"([ACGT])\1{5}", sequence) is None for sequence in sequences]


designs = generate_designs(
    model, length=50, initial=seeds, objective="maximize",
    constraint=no_homopolymer6,
    output_dir=run / "designs",
)
designs.groupby("algorithm")["score"].agg(["size", "median", "max"])


,size,median,max
algorithm,,,
adalead,100,9.197912,9.484251
basesearch,100,8.834576,9.082095
beam_search,100,9.321070,9.762297
cross_entropy_method,100,9.366937,9.484161
den,100,9.126588,9.315540
fast_seqprop,100,8.936005,9.175865
genetic_algorithm,100,9.468585,9.616087
ism_greedy,100,9.322851,9.639274
ledidi,100,8.765347,9.005521




## 3. Discover motifs

Extract positive seqlets from in silico mutagenesis scores and discover motifs with XSTREME.



In [3]:
motifs = call_motifs(
    designs, model=model, reverse_complement=False,
    output_dir=run / "motifs",
)
motifs.support.query("retained")[["motif_id", "seqlets", "parents", "sites_on_unique_DNA"]]


,motif_id,seqlets,parents,sites_on_unique_DNA
0,1-AUAUAG,20,20,184.0
1,2-ARGAAUUG,33,33,165.0
2,3-AUCUAAUU,12,12,921.0
3,5-AUAUUGUUR,38,37,452.0
6,13-AUACUGAA,10,10,390.0
8,17-AAUUAUU,15,15,811.0
15,8-AAUUGAAUUGU,13,13,498.0




## 4. Select a panel



In [4]:
gate = float(designs.drop_duplicates("sequence").score.quantile(0.2))
selected = select_candidates(designs, motifs=motifs, top_n=10, min_score=gate, output_dir=run / "selected")
selected[["selection_rank", "sequence", "score", "present_motifs"]]


,selection_rank,sequence,score,present_motifs
0,1,TTGATTTTAATTTAATTGATATAGATTTTATTGTTTTTAAATTAAT...,9.762297,1-AUAUAG;3-AUCUAAUU;5-AUAUUGUUR;17-AAUUAUU;8-A...
155,2,CCCAAAAACCTTATTTGAGTAAATTCTTGATAGTGTTTAGAATATT...,9.431980,13-AUACUGAA
873,3,GAAAGAATTGCTAATCGGTAGACTAAATCATTAGGTACGCGAAGAA...,8.770403,2-ARGAAUUG
218,4,GTTCTTTGTAATAGTTTTCTTTTTATCTTCTGTACTTATTTAAATT...,9.370036,3-AUCUAAUU
696,5,GAAATAGTCTATAGATCATTCGTATAGTAGATTTAGTTAGAAATAT...,8.904444,1-AUAUAG
86,6,TTGAAGTGTTATTGTTCTTGATTGATCTGTTTGAAATACTTGAAAT...,9.475491,8-AAUUGAAUUGU
858,7,CTTGCCGAATAATTTCCATATTGCTAGTTATTACTACTTTACTTAA...,8.792570,2-ARGAAUUG;13-AUACUGAA;17-AAUUAUU
59,8,TTGAACTTAGATTGATTTTTAGTATTTTAATATTTAATTTTAGAAA...,9.501159,3-AUCUAAUU;13-AUACUGAA;8-AAUUGAAUUGU
309,9,TTTCATTTTAGTAATATTGTTGTTATATTCTGAACTTAGTTAACAT...,9.320565,5-AUAUUGUUR;13-AUACUGAA
515,10,GTTTGTTCATATCTATAATTTTTACGTTAATAGTTAAATTAAAAAT...,9.132324,17-AAUUAUU
